In [1]:
import pandas as pd
import os

In [166]:
# Function to calculate cost
def calculate_cost(row):
    # Base cost per km
    cost = row['Distance [km]'] * cost_factor_transportation
    
    # Add additional costs for Suez or Panama
    if pd.notna(row['Suez or Panama']):
        if 'Suez' in row['Suez or Panama']:
            cost += cost_factor_suez
        elif 'Panama' in row['Suez or Panama']:
            cost += cost_factor_panama
    
    return cost

# Function to create the new dataframe based on the "From" column for liquefaction cost
def create_LNG_liquefaction_df(df, factor):
    # Step 1: Create a new dataframe with unique "From" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['From'].unique(), columns=['From'])
    
    # Step 2: Adjust the "From" column by removing "_LNG" and place the original "From" values in the "To" column
    unique_from_df['To'] = unique_from_df['From']
    unique_from_df['From'] = unique_from_df['From'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

# Function to create the new dataframe based on the "To" column for regasification cost
def create_LNG_regasification_df(df, factor):
    # Step 1: Create a new dataframe with unique "To" values and the corresponding "To" column
    unique_from_df = pd.DataFrame(df['To'].unique(), columns=['To'])
    
    # Step 2: Adjust the "To" column by removing "_LNG" and place the original "To" values in the "From" column
    unique_from_df['From'] = unique_from_df['To']
    unique_from_df['To'] = unique_from_df['To'].str.replace('_LNG', '')
    
    # Step 3: Add the "Cost" column with the value of cost_factor_liquification
    unique_from_df['Cost'] = factor
    
    return unique_from_df

#Define a function to multiply the distance by a cost factor for the pipelines
def pipeline_transport_cost(df, cost_factor):
    df["Cost"] = df["distance [km]"] * cost_factor
    return df

# Function to create df_supply_demand_global
def create_supply_demand_df(df, supply=True):
    # Create the new dataframe with required columns
    df_supply_demand_global = pd.DataFrame({
        'Commodity': ['Methane'] * len(df),  # Set 'Methane' for all rows
        'Node': df['Country'] + ('_Prod' if supply else ''),  # Append '_Prod' if supply is True
        'Supply': df['GWh [2020]']  # Use 'GWh [2020]' for supply
    })
    
    return df_supply_demand_global

def expand_with_hydrogen(df, hydrogen_investment):
    if hydrogen_investment:
        return df  # If investment is allowed, return the original dataframe unchanged

    # Check if the input dataframe follows the (Commodity, Source, Destination) structure
    if {'Commodity', 'Source', 'Destination'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Source', 'Destination']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
        
    # If the input dataframe follows the (Commodity, Node, Supply) structure
    elif {'Commodity', 'Node', 'Supply'}.issubset(df.columns):
        # Copy relevant columns and create a hydrogen version
        hydrogen_df = df[['Commodity', 'Node']].copy()
        hydrogen_df['Commodity'] = 'Hydrogen'  # Replace Commodity with Hydrogen
    
    else:
        raise ValueError("Unexpected dataframe format. Must contain either ['Commodity', 'Source', 'Destination'] or ['Commodity', 'Node', 'Supply']")

    # Fill all other columns with 0
    for col in df.columns:
        if col not in hydrogen_df.columns:  # Skip the required columns
            hydrogen_df[col] = 0

    # Combine the original dataframe with the new hydrogen dataframe
    df_expanded = pd.concat([df, hydrogen_df], ignore_index=True)

    return df_expanded

def integrate_pipeline_costs(df_pipelines, df_transport_cost):
    """
    Merges transport cost data into the pipeline dataframe based on matching From-To relationships, 
    considering both directions (From->To and To->From).
    
    Parameters:
        df_pipelines (pd.DataFrame): DataFrame containing pipeline capacities.
        df_transport_cost (pd.DataFrame): DataFrame containing transport distances and costs.
        
    Returns:
        pd.DataFrame: Updated pipeline DataFrame with an additional 'cost' column.
    """
    # Create reversed pairs for bidirectional matching (From -> To and To -> From)
    df_reversed = df_transport_cost.rename(columns={'From': 'To', 'To': 'From', 'Cost': 'Cost_reversed'})
    
    # Concatenate original and reversed cost data to handle both directions
    df_cost = pd.concat([df_transport_cost[['From', 'To', 'Cost']], df_reversed[['From', 'To', 'Cost_reversed']]], ignore_index=True)
    
    # Merge the concatenated cost data with df_pipelines to get the corresponding cost
    df_pipelines = df_pipelines.merge(
        df_cost, 
        on=['From', 'To'], 
        how='left'
    )
    
    # For cases where the reverse relation exists, use the reversed cost value
    df_pipelines['Cost'] = df_pipelines['Cost'].fillna(df_pipelines['Cost_reversed'])

    # Drop the extra reversed cost column (no longer needed)
    df_pipelines = df_pipelines.drop(columns=['Cost_reversed'])
    
    return df_pipelines

In [312]:
def create_edges_cap_cost_dataframe(df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df, 
                                    df_global_pipe_transport_cost, df_pipelines_europe, df_LNG_europe):
    # Step 1: Collect unique From-To pairs from all dataframes
    unique_pairs = set()

    # Collecting unique pairs from each dataframe
    for df in [df_cost_per_km, LNG_regasification_df, LNG_liquefaction_df, 
               df_global_pipe_transport_cost, df_pipelines_europe, df_LNG_europe]:
        for _, row in df.iterrows():
            unique_pairs.add((row['From'], row['To']))

    # Step 2: Create the new structured dataframe with Source and Destination columns
    df_all_cost = pd.DataFrame(unique_pairs, columns=['Source', 'Destination'])

    # Step 3: Add required columns
    df_all_cost.insert(0, 'Commodity', 'Methane')  # Set commodity as Methane

    # Step 4: Merge cost and capacity data

    # Merge with df_cost_per_km (Cost)
    df_all_cost = df_all_cost.merge(
        df_cost_per_km[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_cost_per_km'})

    # Merge with LNG_regasification_df (Cost)
    df_all_cost = df_all_cost.merge(
        LNG_regasification_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_regasification'})

    # Merge with LNG_liquefaction_df (Cost)
    df_all_cost = df_all_cost.merge(
        LNG_liquefaction_df[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_liquefaction'})

    # Merge with df_global_pipe_transport_cost (Cost)
    df_all_cost = df_all_cost.merge(
        df_global_pipe_transport_cost[['From', 'To', 'Cost']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_global_pipe_transport'})

    # Merge with df_pipelines_europe (Cost and GWh/a)
    df_all_cost = df_all_cost.merge(
        df_pipelines_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_pipelines_europe', 'GWh/a': 'GWh_a_pipelines_europe'})

    # ✅ Merge with df_LNG_europe (Cost and GWh/a)
    df_all_cost = df_all_cost.merge(
        df_LNG_europe[['From', 'To', 'Cost', 'GWh/a']],
        left_on=['Source', 'Destination'],
        right_on=['From', 'To'],
        how='left'
    ).drop(columns=['From', 'To']).rename(columns={'Cost': 'Cost_LNG_europe', 'GWh/a': 'GWh_a_LNG_europe'})

    # ✅ Step 5: Set costs_edge (sum available cost values)
    df_all_cost['costs_edge'] = df_all_cost[
        ['Cost_cost_per_km', 'Cost_LNG_regasification', 'Cost_LNG_liquefaction', 
         'Cost_global_pipe_transport', 'Cost_pipelines_europe', 'Cost_LNG_europe']
    ].sum(axis=1, min_count=1)

    # ✅ Step 6: Set correct capacities
    # If pipeline/LNG capacity exists, use it. Otherwise, keep 9999.
    df_all_cost['initial_capacities'] = df_all_cost['GWh_a_pipelines_europe'].fillna(0) + df_all_cost['GWh_a_LNG_europe'].fillna(0)
    df_all_cost['max_capacities'] = df_all_cost['initial_capacities']

    # If capacity is still 0 (no data), set to 9999
    df_all_cost.loc[df_all_cost['initial_capacities'] == 0, 'initial_capacities'] = 9999
    df_all_cost.loc[df_all_cost['max_capacities'] == 0, 'max_capacities'] = 9999

    # ✅ Step 7: Drop unnecessary columns
    df_all_cost = df_all_cost.drop(columns=[
        'Cost_cost_per_km', 'Cost_LNG_regasification', 'Cost_LNG_liquefaction', 
        'Cost_global_pipe_transport', 'Cost_pipelines_europe', 'Cost_LNG_europe',
        'GWh_a_pipelines_europe', 'GWh_a_LNG_europe'
    ])

    # ✅ Step 8: Add empty columns for future values
    df_all_cost['new_build_cost'] = 1000000
    df_all_cost['conversion_cost'] = 0
    df_all_cost['conversion_capacity_factor'] = 1

    return df_all_cost

In [313]:
hydrogen_investment=False

cost_factor_transportation = 2
cost_factor_liquefaction = 2000
cost_factor_regasification = 2000
cost_factor_panama = 2000
cost_factor_suez = 2000

cost_factor_pipelines = 0.5

## Import Data

In [314]:
# Specify the path to your Excel file
input_file_path_1 = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = '\\Gas_Import_Russian_Invasion.xlsx'
# Specify the path to your Excel file
input_file_path_2 = os.path.join('..', '..','00_code_base', '07_data_prep')
distances_file_name = '\\distances.xlsx'

input_file_path_1  = input_file_path_1 + excel_file_name
full_input_path_1 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_1))
input_file_path_2  = input_file_path_2 + distances_file_name
full_input_path_2 = os.path.abspath(os.path.join(os.getcwd(), input_file_path_2))

In [315]:
df_LNG_global = pd.read_excel(full_input_path_1, sheet_name='global_LNG_connections')
df_LNG_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_lng_europe_2020')
df_production_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_europe_2020')
df_consumption_europe = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_europe_2020')
df_production_global = pd.read_excel(full_input_path_1, sheet_name='nodes_gas_prod_world_2020')
df_consumption_global = pd.read_excel(full_input_path_1, sheet_name='nodes_demand_world_2020')
df_pipelines_europe = pd.read_excel(full_input_path_1, sheet_name='Connections_2021')
df_pipelines_global = pd.read_excel(full_input_path_1, sheet_name='Global_Connections')

#load input for inner-European distances
df_distances = pd.read_excel(full_input_path_2, sheet_name='distances')
#Drop the "Unnamed: 0" column in df_distances
df_distances.drop(columns=["Unnamed: 0"], inplace=True)

In [316]:
#adjust the data frames and remove unnecessary content
#gobal demand
df_consumption_global = df_consumption_global.iloc[:-3]
df_consumption_global = df_consumption_global.iloc[:, :-2]
#gobal production
df_production_global = df_production_global.iloc[:-3]
#globale pipelines
df_pipelines_global = df_pipelines_global.iloc[:, :-5]
#europe demand
df_consumption_europe = df_consumption_europe.iloc[:-11]
#europe production
df_production_europe = df_production_europe.iloc[:-7]
df_production_europe = df_production_europe.iloc[:, :-7]
#europe LNG
df_LNG_europe = df_LNG_europe.iloc[:, :-6]
#add a From column to the European LNG data
df_LNG_europe.insert(1, 'From', df_LNG_europe['To'] + '_LNG')
#LNG global
df_LNG_global = df_LNG_global.iloc[:, :-5]

#adjust naming
df_distances = df_distances.rename(columns={"from [NUTS_ID]": "From"})
df_distances = df_distances.rename(columns={"to [NUTS_ID]": "To"})

In [317]:
df_LNG_global

,From,To,GWh/d [capacity],GWh/a,TWh/a,Distance [miles],Distance [km],Suez or Panama
0,USA_LNG,SA_LNG,NaN,NaN,0,5323,8566.540762,NaN
1,USA_LNG,AF_LNG,NaN,NaN,0,6194,9968.279820,NaN
2,USA_LNG,CN_LNG,NaN,NaN,0,10194,16405.657812,Panama
3,USA_LNG,JS_LNG,NaN,NaN,0,9266,14912.186117,Panama
4,USA_LNG,IN_LNG,NaN,NaN,0,9568,15398.208156,Suez
...,...,...,...,...,...,...,...,...
209,RU_LNG,EE_LNG,NaN,NaN,0,3259,5244.853719,NaN
210,RU_LNG,FI_LNG,NaN,NaN,0,3259,5244.853719,NaN
211,RU_LNG,DE_LNG,NaN,NaN,0,2730,4393.510479,NaN
212,RU_LNG,IE_LNG,NaN,NaN,0,3077,4951.953020,NaN


In [318]:
df_LNG_europe

,LNG,From,To,LNG Capacity Mrd m3 [2020],Imports [2020],Share [2020],Capacity after grid connection [2020],Limitation,GWh/a,Limitation in TWh
0,Belgium,BE_LNG,BE,11.40,5.1,0.447368,9.0,11.40,111378.0,111.3780
1,France,FR_LNG,FR,33.00,19.6,0.593939,33.0,33.00,322410.0,322.4100
2,Greece,EL_LNG,EL,7.00,2.5,0.357143,7.5,7.50,73275.0,73.2750
3,Italy,IT_LNG,IT,15.95,12.1,0.758621,15.8,15.95,155831.5,155.8315
4,Croatia,HR_LNG,HR,2.60,0.4,0.153846,2.6,2.60,25402.0,25.4020
5,Lithuania,LT_LNG,LT,4.00,1.6,0.400000,4.0,4.00,39080.0,39.0800
6,Netherlands,NL_LNG,NL,12.00,8.2,0.683333,12.5,12.50,122125.0,122.1250
7,Poland,PL_LNG,PL,6.20,4.1,0.661290,5.8,6.20,60574.0,60.5740
8,Portugal,PT_LNG,PT,7.60,6.1,0.802632,7.3,7.60,74252.0,74.2520
9,Spain,ES_LNG,ES,60.10,20.9,0.347754,67.1,67.10,655567.0,655.5670


### Demand and Supply input sheet

In [319]:
#demand Europe
df_demand_europe = create_supply_demand_df(df_consumption_europe, supply=False)
#supply Europe
df_supply_europe = create_supply_demand_df(df_production_europe)

In [320]:
#demand Europe
df_demand_global = create_supply_demand_df(df_consumption_global, supply=False)
#supply global
df_supply_global = create_supply_demand_df(df_production_global)

In [321]:
# Combine the demand and supply data frames
df_all_supply_demand = pd.concat([df_demand_europe, df_supply_europe, df_demand_global, df_supply_global], ignore_index=True)

In [322]:
# Apply the function to each row and create a new column 'Cost per km'
df_LNG_global['Cost'] = df_LNG_global.apply(calculate_cost, axis=1)

# Create a new dataframe with the relevant columns
df_cost_per_km = df_LNG_global[['From', 'To', 'Distance [km]', 'Suez or Panama', 'Cost']]

In [323]:
# Get LNG liquefaction cost df
LNG_liquefaction_df = create_LNG_liquefaction_df(df_LNG_global, cost_factor_liquefaction)
# Get LNG regasification cost df
LNG_regasification_df = create_LNG_regasification_df(df_LNG_global, cost_factor_regasification)

In [324]:
#add regasification cost to European LNG import nodes
df_LNG_europe.insert(3, "Cost", cost_factor_regasification)

In [325]:
#Pipeline cost
#calculate European pipeline cost
df_european_pipe_transport_cost = pipeline_transport_cost(df_distances, cost_factor_pipelines)
df_global_pipe_transport_cost = pipeline_transport_cost(df_pipelines_global, cost_factor_pipelines)

In [326]:
# Example usage:
df_pipelines_europe = integrate_pipeline_costs(df_pipelines_europe, df_european_pipe_transport_cost)

In [327]:
# Apply the create_edges_cap_cost_dataframe function to get edges input 
df_all_cost = create_edges_cap_cost_dataframe(
    df_cost_per_km, 
    LNG_regasification_df, 
    LNG_liquefaction_df, 
    df_global_pipe_transport_cost, 
    df_pipelines_europe, 
    df_LNG_europe
)

# Display the result
df_edges_cap_cost

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,EG_LNG,IE_LNG,9999.0,9999.0,9685.035188,1000000,0,1
1,Methane,ME_LNG,PL_LNG,9999.0,9999.0,24379.544588,1000000,0,1
2,Methane,RU_LNG,LT_LNG,9999.0,9999.0,9807.345370,1000000,0,1
3,Methane,AS_LNG,AS,9999.0,9999.0,2000.000000,1000000,0,1
4,Methane,RU_LNG,FI_LNG,9999.0,9999.0,10489.707437,1000000,0,1
...,...,...,...,...,...,...,...,...,...
275,Methane,ES,ES_LNG,9999.0,9999.0,2000.000000,1000000,0,1
276,Methane,CN,CN_LNG,9999.0,9999.0,2000.000000,1000000,0,1
277,Methane,IE,IE_LNG,9999.0,9999.0,2000.000000,1000000,0,1
278,Methane,FR,FR_LNG,9999.0,9999.0,2000.000000,1000000,0,1


In [296]:
def ensure_direct_connections(df, df_LNG_europe, cost_factor_liquefaction):
    # Ensure no NaN values in 'Source'
    df = df.dropna(subset=["Source"])

    # Extract all LNG sources from df_edges_cap_cost
    lng_sources = set(df[df["Source"].astype(str).str.endswith("_LNG")]["Source"])

    # Extract LNG sources that should be **excluded** (those in df_LNG_europe["From"])
    excluded_lng_sources = set(df_LNG_europe["From"].dropna().unique())

    # Keep only LNG sources that **should be considered**
    valid_lng_sources = lng_sources - excluded_lng_sources

    # Extract base country codes that need new connections
    base_countries = {src.replace("_LNG", "") for src in valid_lng_sources}

    # Find missing connections
    missing_rows = []
    for country in base_countries:
        source = country
        destination = f"{country}_LNG"

        # Ensure this direct connection doesn't already exist
        if not ((df["Source"] == source) & (df["Destination"] == destination)).any():
            missing_rows.append({
                "Commodity": "Methane",
                "Source": source,
                "Destination": destination,
                "initial_capacities": 9999,
                "max_capacities": 9999,
                "costs_edge": cost_factor_liquefaction,
                "new_build_cost": 1000000,
                "conversion_cost": 0,
                "conversion_capacity_factor": 1
            })

    # Append missing rows if needed
    if missing_rows:
        df = pd.concat([df, pd.DataFrame(missing_rows)], ignore_index=True)

    return df

In [297]:
#add missing connections to LNG terminals
df_edges_cap_cost = ensure_direct_connections(df_edges_cap_cost, df_LNG_europe, cost_factor_liquefaction)

In [298]:
df_edges_cap_cost

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,EG_LNG,IE_LNG,9999.0,9999.0,9685.035188,1000000,0,1
1,Methane,ME_LNG,PL_LNG,9999.0,9999.0,24379.544588,1000000,0,1
2,Methane,RU_LNG,LT_LNG,9999.0,9999.0,9807.345370,1000000,0,1
3,Methane,AS_LNG,AS,9999.0,9999.0,2000.000000,1000000,0,1
4,Methane,RU_LNG,FI_LNG,9999.0,9999.0,10489.707437,1000000,0,1
...,...,...,...,...,...,...,...,...,...
275,Methane,ES,ES_LNG,9999.0,9999.0,2000.000000,1000000,0,1
276,Methane,CN,CN_LNG,9999.0,9999.0,2000.000000,1000000,0,1
277,Methane,IE,IE_LNG,9999.0,9999.0,2000.000000,1000000,0,1
278,Methane,FR,FR_LNG,9999.0,9999.0,2000.000000,1000000,0,1


### check for hydrogen and append with zeros if no repurpose investigation

In [328]:
# Example: Calling function with hydrogen_investment = False
df_edges_complete = expand_with_hydrogen(df_edges_cap_cost, hydrogen_investment)

# Display the result
df_edges_complete

,Commodity,Source,Destination,initial_capacities,max_capacities,costs_edge,new_build_cost,conversion_cost,conversion_capacity_factor
0,Methane,EG_LNG,IE_LNG,9999.0,9999.0,9685.035188,1000000,0,1
1,Methane,ME_LNG,PL_LNG,9999.0,9999.0,24379.544588,1000000,0,1
2,Methane,RU_LNG,LT_LNG,9999.0,9999.0,9807.345370,1000000,0,1
3,Methane,AS_LNG,AS,9999.0,9999.0,2000.000000,1000000,0,1
4,Methane,RU_LNG,FI_LNG,9999.0,9999.0,10489.707437,1000000,0,1
...,...,...,...,...,...,...,...,...,...
555,Hydrogen,ES,ES_LNG,0.0,0.0,0.000000,0,0,0
556,Hydrogen,CN,CN_LNG,0.0,0.0,0.000000,0,0,0
557,Hydrogen,IE,IE_LNG,0.0,0.0,0.000000,0,0,0
558,Hydrogen,FR,FR_LNG,0.0,0.0,0.000000,0,0,0


In [329]:
# Example: Calling function with hydrogen_investment = False
df_demand_supply_complete = expand_with_hydrogen(df_all_supply_demand, hydrogen_investment)

# Display the result
df_demand_supply_complete

,Commodity,Node,Supply
0,Methane,AL,-7717.933443
1,Methane,AT,-83311.215033
2,Methane,BE,-166072.218600
3,Methane,BA,-9469.790436
4,Methane,BG,-28534.750500
...,...,...,...
231,Hydrogen,IN_Prod,0.000000
232,Hydrogen,AS_Prod,0.000000
233,Hydrogen,CR_Prod,0.000000
234,Hydrogen,EG_Prod,0.000000


# export

In [330]:
df_edges_complete.to_excel("inputs.xlsx", index=False)